In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, make_scorer

In [10]:
pd.set_option("display.max_columns", None)

In [11]:
df=pd.read_csv("german_credit_data.csv")

In [12]:
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

In [13]:
df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,49,male,1,own,little,NaN,2096,12,education,good
3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,53,male,2,free,little,little,4870,24,car,bad


In [14]:
# Как са разпределени
df["Risk"].value_counts()

Risk
good    700
bad     300
Name: count, dtype: int64

In [15]:
df['Saving accounts'] = df['Saving accounts'].fillna('Unknown')
df['Checking account'] = df['Checking account'].fillna('Unknown')

df['Sex'] = df['Sex'].str.strip().map({'male': 0, 'female': 1})
df['Housing'] = df['Housing'].str.strip().map({'own': 0, 'free': 1, 'rent': 2})
df['Risk'] = df['Risk'].str.strip().map({'good': 0, 'bad': 1})
df = df.dropna(subset=['Risk'])

label_encoders = {}
for col in ['Saving accounts', 'Checking account', 'Purpose']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le


X = df.drop('Risk', axis=1)
y = df['Risk']

# ДЕФИНИРАНЕ НА МЕТРИКИ (F1 и F2)
f2_scorer = make_scorer(fbeta_score, beta=2)
scoring_metrics = {'F1': 'f1', 'F2': f2_scorer}

#КРОС-ВАЛИДАЦИЯ И МОДЕЛИ С БАЛАНСИРАНЕ
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

#тежестта за XGBoost (700 добри / 300 лоши = 2.33)
ratio = (y == 0).sum() / (y == 1).sum()

models = {
    #class_weight='balanced' за автоматична тежест на класовете
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', class_weight='balanced')),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),

    # При XGBoost параметъра scale_pos_weight
    "XGBoost": XGBClassifier(eval_metric='logloss', scale_pos_weight=ratio, random_state=42)
}

# ИЗПЪЛНЕНИЕ И ПРИНТИРАНЕ НА РЕЗУЛТАТИТЕ
print(f"Общ брой записи за анализ: {len(df)}")
print("-" * 50)

for name, model in models.items():
    cv_results = cross_validate(model, X, y, scoring=scoring_metrics, cv=cv, n_jobs=-1)

    f1_mean = np.mean(cv_results['test_F1'])
    f1_std = np.std(cv_results['test_F1'])
    f2_mean = np.mean(cv_results['test_F2'])
    f2_std = np.std(cv_results['test_F2'])

    print(f"Модел: {name}")
    print(f"   F1-Score (Баланс):    {f1_mean:.4f} (+/- {f1_std:.4f})")
    print(f"   F2-Score (Сигурност): {f2_mean:.4f} (+/- {f2_std:.4f})")
    print("-" * 30)

Общ брой записи за анализ: 1000
--------------------------------------------------
Модел: Logistic Regression
   F1-Score (Баланс):    0.4992 (+/- 0.0592)
   F2-Score (Сигурност): 0.5565 (+/- 0.0670)
------------------------------
Модел: Decision Tree
   F1-Score (Баланс):    0.4539 (+/- 0.0704)
   F2-Score (Сигурност): 0.4474 (+/- 0.0833)
------------------------------
Модел: Random Forest
   F1-Score (Баланс):    0.4842 (+/- 0.0670)
   F2-Score (Сигурност): 0.4248 (+/- 0.0691)
------------------------------
Модел: XGBoost
   F1-Score (Баланс):    0.5455 (+/- 0.0607)
   F2-Score (Сигурност): 0.5421 (+/- 0.0733)
------------------------------


In [16]:
import joblib
final_model = XGBClassifier(eval_metric='logloss', scale_pos_weight=ratio, random_state=42)
final_model.fit(X, y)

joblib.dump(final_model, 'credit_model.pkl')
joblib.dump(label_encoders, 'label_encoder.pkl')

print("\n--- ГОТОВО! ---")
print("Моделът е запазен като 'credit_model.pkl'. Вече можете да стартирате app.py.")


--- ГОТОВО! ---
Моделът е запазен като 'credit_model.pkl'. Вече можете да стартирате app.py.
